# Natural Language Processing - Text Preprocessing

## Libraries and settings

In [63]:
# Libraries
import os
import re
import string
import numpy as np
import pandas as pd
from pprint import pprint

import nltk

# Import only once
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.chunk import tree2conlltags
from nltk.chunk import conlltags2tree
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Current working directory
print('Current working directory:', os.getcwd())

Current working directory: /workspaces/data_analytics/Week_11


[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/vscode/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/vscode/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/vscode/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## Defining documents

In [64]:
# Defining documents (=sentenses)
d1 = 'The supervisor approves the shift swap request.'
d2 = 'Agents schedule weekend shifts and track breaks.'
d3 = 'The system notifies employees and updates the calendar.'

corpus_01 = d1 + ' ' + d2 + ' ' + d3
corpus_01

'The supervisor approves the shift swap request. Agents schedule weekend shifts and track breaks. The system notifies employees and updates the calendar.'

## Text preprocessing
#### Steps:
- Text to lowercase
- Removing punctuations
- Tokenization
- Removal of stop words
- Lemmatization

### Text to lowercase

In [65]:
# Text to lowercase function
def text_lowercase(text):
    return text.lower()

# Text to lowercase
corpus_02 = text_lowercase(corpus_01)
corpus_02

'the supervisor approves the shift swap request. agents schedule weekend shifts and track breaks. the system notifies employees and updates the calendar.'

### Removing punctuation

In [66]:
# Remove punctuation function
def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

# Remove punctuation
corpus_03 = remove_punctuation(corpus_02)
corpus_03

'the supervisor approves the shift swap request agents schedule weekend shifts and track breaks the system notifies employees and updates the calendar'

### Tokenize text & removal of stopwords

In [67]:
# Show english stopwords
eng_stopwords = set(stopwords.words('english'))
print("List of english stopwords:")
print(eng_stopwords)

List of english stopwords:
{'between', 'why', "she'll", 'for', 'same', "isn't", 'shan', "hadn't", 'few', 'our', 'she', 'ourselves', 'down', 'through', 've', "i'm", "they've", 'no', "we'd", 'some', 'than', 'only', 'when', 'were', 'who', 'both', "they'll", "we'll", 'an', 'own', 'your', 'there', 'yourselves', "they'd", "we're", 'with', 'off', "he'll", 'into', 'y', 'been', 'won', "weren't", 'you', 'he', 's', 'hers', 'below', 'be', 'after', "you'd", "i've", 'is', 'did', 'hasn', 'me', 'this', "she's", "we've", 'what', "you're", 'ain', 'more', 'too', 'do', 'am', "you've", 'whom', 'by', 'how', 'not', 'out', 't', 'about', 'ours', 'again', "don't", 'other', "shan't", "shouldn't", 'd', 'itself', 'these', 'having', 'above', 'they', 'because', 'its', 'o', 'mustn', 'doesn', 'i', 'or', 'that', "doesn't", 'theirs', 'him', 'where', 'during', "should've", 'such', 'so', 'a', 'mightn', 'from', 'will', 're', 'yours', 'once', 'just', 'have', 'those', "you'll", "couldn't", 'at', 'hadn', "he's", 'in', 'had', 

In [68]:
# Function for tokenization and the removal of stopwords
def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    word_tokens = word_tokenize(text)
    filtered_text = [word for word in word_tokens if word not in stop_words]
    return filtered_text
 
# Remove stopwords
corpus_04 = remove_stopwords(corpus_03)
print(corpus_04, end="")

['supervisor', 'approves', 'shift', 'swap', 'request', 'agents', 'schedule', 'weekend', 'shifts', 'track', 'breaks', 'system', 'notifies', 'employees', 'updates', 'calendar']

### Lemmatization

In [69]:
# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

# Lemmatize string function
def lemmatize_word(text):
    word_tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(word, pos ='v') for word in word_tokens]
    return lemmas

# Lemmatize
lem = []
for i in corpus_04:
    lem.append(lemmatize_word(i))

# Nested list to list
corpus_05 = [' '.join([str(x) for x in lst]) for lst in lem]

print('Before lemmatization:')
print(corpus_04, '\n')

print('After lemmatization:')
print(corpus_05, end="")

Before lemmatization:
['supervisor', 'approves', 'shift', 'swap', 'request', 'agents', 'schedule', 'weekend', 'shifts', 'track', 'breaks', 'system', 'notifies', 'employees', 'updates', 'calendar'] 

After lemmatization:
['supervisor', 'approve', 'shift', 'swap', 'request', 'agents', 'schedule', 'weekend', 'shift', 'track', 'break', 'system', 'notify', 'employees', 'update', 'calendar']

## Redefine the text corpus (pre-processed)

In [70]:
# We will use the lemmatized words above to re-define our corpus 
corpus = [  
    'supervisor approve shift swap request',
    'agent schedule weekend shift track break',
    'system notify employee update calendar'
    ]

## Document-term matrix with ngram_range=(1,1)

In [71]:
# Vectorizer with ngram_range=(1,1)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(1,1))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   agent  approve  break  calendar  employee  notify  request  schedule  \
0      0        1      0         0         0       0        1         0   
1      1        0      1         0         0       0        0         1   
2      0        0      0         1         1       1        0         0   

   shift  supervisor  swap  system  track  update  weekend  
0      1           1     1       0      0       0        0  
1      1           0     0       0      1       0        1  
2      0           0     0       1      0       1        0  


## Document-term matrix with ngram_range=(2,2)

In [72]:
# Vectorizer with with ngram_range=(2,2)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(2,2))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   agent schedule  approve shift  employee update  notify employee  \
0               0              1                0                0   
1               1              0                0                0   
2               0              0                1                1   

   schedule weekend  shift swap  shift track  supervisor approve  \
0                 0           1            0                   1   
1                 1           0            1                   0   
2                 0           0            0                   0   

   swap request  system notify  track break  update calendar  weekend shift  
0             1              0            0                0              0  
1             0              0            1                0              1  
2             0              1            0                1              0  


## Term frequency-inverse document frequency (TF-IDF)
- For details see: https://www.learndatasci.com/glossary/tf-idf-term-frequency-inverse-document-frequency

### Term Frequency (TF)

In [73]:
# Compute Term Frequency (TF)
words_set = set()
for doc in corpus:
    words = doc.split(' ')
    words_set = words_set.union(set(words))
    
print('Number of words in the corpus:',len(words_set), '\n')
print('The words in the corpus: \n', words_set)

# Number of documents in the corpus
n_docs = len(corpus)

# Number of unique words in the corpus 
n_words_set = len(words_set)

df_tf = pd.DataFrame(np.zeros((n_docs, n_words_set)), 
                     columns=list(words_set))

print("\nTerm Frequency (TF):")
for i in range(n_docs):
    # Words in the document
    words = corpus[i].split(' ')
    for w in words:
        df_tf[w][i] = df_tf[w][i] + (1 / len(words))
        
print(df_tf.round(4))

Number of words in the corpus: 15 

The words in the corpus: 
 {'update', 'weekend', 'supervisor', 'agent', 'break', 'swap', 'calendar', 'notify', 'track', 'request', 'system', 'shift', 'employee', 'schedule', 'approve'}

Term Frequency (TF):
   update  weekend  supervisor   agent   break  swap  calendar  notify  \
0     0.0   0.0000         0.2  0.0000  0.0000   0.2       0.0     0.0   
1     0.0   0.1667         0.0  0.1667  0.1667   0.0       0.0     0.0   
2     0.2   0.0000         0.0  0.0000  0.0000   0.0       0.2     0.2   

    track  request  system   shift  employee  schedule  approve  
0  0.0000      0.2     0.0  0.2000       0.0    0.0000      0.2  
1  0.1667      0.0     0.0  0.1667       0.0    0.1667      0.0  
2  0.0000      0.0     0.2  0.0000       0.2    0.0000      0.0  


### Inverse Document Frequency (IDF)

In [74]:
# Computing Inverse Document Frequency (IDF)
print("\nInverse Document Frequency (IDF):")

idf = {}

for w in words_set:
    
    # k = number of documents that contain this word
    k = 0
    
    for i in range(n_docs):
        if w in corpus[i].split():
            k += 1
            
    idf[w] =  np.log10(n_docs / k).round(4)
    
    print(f'{w:>15}: {idf[w]:>10}')


Inverse Document Frequency (IDF):
         update:     0.4771
        weekend:     0.4771
     supervisor:     0.4771
          agent:     0.4771
          break:     0.4771
           swap:     0.4771
       calendar:     0.4771
         notify:     0.4771
          track:     0.4771
        request:     0.4771
         system:     0.4771
          shift:     0.1761
       employee:     0.4771
       schedule:     0.4771
        approve:     0.4771


### Term Frequency - Inverse Document Frequency (TF-IDF)

In [75]:
# Computing TF-IDF
df_tf_idf = df_tf.copy()

for w in words_set:
    for i in range(n_docs):
        df_tf_idf[w][i] = df_tf[w][i] * idf[w]

print('\nTF-IDF:')
print(df_tf_idf.round(4))


TF-IDF:
   update  weekend  supervisor   agent   break    swap  calendar  notify  \
0  0.0000   0.0000      0.0954  0.0000  0.0000  0.0954    0.0000  0.0000   
1  0.0000   0.0795      0.0000  0.0795  0.0795  0.0000    0.0000  0.0000   
2  0.0954   0.0000      0.0000  0.0000  0.0000  0.0000    0.0954  0.0954   

    track  request  system   shift  employee  schedule  approve  
0  0.0000   0.0954  0.0000  0.0352    0.0000    0.0000   0.0954  
1  0.0795   0.0000  0.0000  0.0294    0.0000    0.0795   0.0000  
2  0.0000   0.0000  0.0954  0.0000    0.0954    0.0000   0.0000  


## Part-of-Speach (POS) tagging
For meaning of POS-tags see: https://pythonexamples.org/nltk-pos-tagging

In [76]:
text = '''The supervisor quickly approved two shift swaps for Anna on Friday.'''

def preprocess(sent):
    sent = nltk.word_tokenize(sent)
    sent = nltk.pos_tag(sent)
    return sent

sent = preprocess(text)
pattern = 'NP: {<DT>?<JJ>*<NN>}'

cp = nltk.RegexpParser(pattern)
cs = cp.parse(sent)

iob_tagged = tree2conlltags(cs)

# Print the POS-tags
pprint(iob_tagged)

[('The', 'DT', 'O'),
 ('supervisor', 'JJ', 'O'),
 ('quickly', 'RB', 'O'),
 ('approved', 'VBD', 'O'),
 ('two', 'CD', 'O'),
 ('shift', 'NN', 'B-NP'),
 ('swaps', 'NNS', 'O'),
 ('for', 'IN', 'O'),
 ('Anna', 'NNP', 'O'),
 ('on', 'IN', 'O'),
 ('Friday', 'NNP', 'O'),
 ('.', '.', 'O')]


## Short explanations (b–f)

### b) Define documents & apply preprocessing
I defined three own example documents and applied the standard preprocessing pipeline: 
lowercasing, punctuation removal, tokenization, stopword removal, and lemmatization. 

This reduces noise and normalizes different word forms so the texts are easier to compare.

### c) Manually redefine the corpus
I created the final `corpus` using the preprocessed and lemmatized words from each document.
Each list entry represents one document as a single space-separated string, which is the input format required for the vectorizers.

### d) Document-term matrices (DTM) with n-grams
Using the corpus from (c), I generated two document-term matrices:
- **Unigrams (ngram_range = (1,1))** to count single words per document
- **Bigrams (ngram_range = (2,2))** to count two-word phrases per document  
This shows how term frequencies change depending on the chosen n-gram size.

### e) TF, IDF and TF-IDF
Based on the corpus from (c), I calculated:
- **TF (Term Frequency):** normalized frequency of each word in a document
- **IDF (Inverse Document Frequency):** how rare/common a word is across all documents (computed as log10(N/df))
- **TF-IDF:** TF × IDF, highlighting words that are frequent in a document but rare in the corpus overall

### f) POS tagging
I chose my own text example and derived POS tags using NLTK. 


In my example text, the following POS tags appear in the output:

DT (Determiner): a word that introduces a noun, e.g., “The”

RB (Adverb): modifies a verb/adjective and often answers “how?”, e.g., “quickly”

VBD (Verb, past tense): a verb in the simple past, e.g., “approved”

CD (Cardinal number): a number, e.g., “two”

NN (Noun, singular): a singular noun, e.g., “shift”

NNS (Noun, plural): a plural noun, e.g., “swaps”

IN (Preposition / subordinating conjunction): a preposition such as “for” or “on”

NNP (Proper noun, singular): a proper name, e.g., “Anna”, “Friday”

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [77]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1030-azure
Datetime: 2025-12-14 13:56:30
Python Version: 3.11.14
-----------------------------------
